2 parts

1. Using Mozilla Data Collective

Authentication
  - API key
  - Accepting the terms for a dataset

Find a dataset, explore its details and download it

2. Finetuning a Whisper model, using Hugging Face

Authentication to Hugging Face

Explore `whisper-tiny`



In [ ]:
import os

mdc_api_key = os.environ.get('MDC_API_KEY')

## Finding a dataset on Mozilla Data Collective

- find one

- agree to use policies

### Request information about the dataset

In [ ]:
import requests

mdc_base_url = "https://mozilladatacollective.com/api"

auth_header = {
    "Authorization": f"Bearer {mdc_api_key}",
    "Content-Type" : "application/json"
}

dataset_id = "cmko7havo02f5nw07rbwwhowe"

request = requests.get(f"{mdc_base_url}/datasets/{dataset_id}", headers=auth_header)

data = request.json()
for key, value in data.items():
    print(f"{key}: {value}")

### Download the dataset

- Request the download link

- download the file

- 'untar' the file

In [ ]:
import requests

request = requests.post(f"{mdc_base_url}/datasets/{dataset_id}/download", headers=auth_header)
data = request.json()


In [5]:
download_url = data.get("downloadUrl")
filename = data.get("filename")

with requests.get(download_url, stream=True) as response:
    response.raise_for_status()
    with open(filename, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)

In [ ]:
import tarfile

download_path = f"mdc_downloads/{filename.split('.')[0]}_extracted_contents"

with tarfile.open(filename, "r:gz") as tar:
    tar.extractall(path=f"./{download_path}")

### Inspect the data

In [7]:
import pandas as pd

common_voice_au_df = pd.read_csv(f"./{download_path}/commonvoice-v24_en-AU/commonvoice-v24_en-AU.csv")

common_voice_au_df = common_voice_au_df[["path","sentence","age","gender","accents","locale","duration_ms"]]
common_voice_au_df["audio"] = common_voice_au_df["path"].apply(lambda path: f"./{download_path}/commonvoice-v24_en-AU/audio_files/{path}")
common_voice_au_df.head()

,path,sentence,age,gender,accents,locale,duration_ms,audio
0,common_voice_en_30513358.mp3,Princess Vilas herself also contributed person...,teens,male_masculine,Australian English,en,7.387500,./mdc_downloads/common-voice-v24-english-en-au...
1,common_voice_en_43618790.mp3,He has also served in the Chamber of Deputies.,NaN,NaN,Australian English,en,6.307500,./mdc_downloads/common-voice-v24-english-en-au...
2,common_voice_en_21099981.mp3,Most of his subjects were found in Devon and C...,thirties,male_masculine,Australian English,en,7.949625,./mdc_downloads/common-voice-v24-english-en-au...
3,common_voice_en_39588772.mp3,Shots rang out as they fled towards the Austri...,NaN,NaN,Australian English,en,5.515500,./mdc_downloads/common-voice-v24-english-en-au...
4,common_voice_en_37211578.mp3,The system is based on electromagnetic induction.,NaN,NaN,Australian English,en,4.055656,./mdc_downloads/common-voice-v24-english-en-au...


### Listen to a sample

In [8]:
from IPython.display import Audio

print(common_voice_au_df["sentence"][0])
sample_mp3_audio = common_voice_au_df["audio"][0]

Audio(sample_mp3_audio)

Princess Vilas herself also contributed personally to the construction of the temple.


## Hugging Face

- install required packages
- authenticate

In [ ]:
!pip install evaluate jiwer transformers

### Load and prep dataset

In [9]:
common_voice_au_df = common_voice_au_df.head(5000)

### Splitting the data into training and testing sets

This takes our collected audio data and divides it into two groups: 80% is set aside for teaching the AI, and the remaining 20% is kept back to test how well it learned — similar to studying from a textbook and then sitting an exam with questions you haven't seen before.

In [10]:
from datasets import load_dataset, DatasetDict, Dataset

common_voice_dataset = Dataset.from_pandas(common_voice_au_df).train_test_split(test_size=0.2)

common_voice_dataset

DatasetDict({
    train: Dataset({
        features: ['path', 'sentence', 'age', 'gender', 'accents', 'locale', 'duration_ms', 'audio'],
        num_rows: 4000
    })
    test: Dataset({
        features: ['path', 'sentence', 'age', 'gender', 'accents', 'locale', 'duration_ms', 'audio'],
        num_rows: 1000
    })
})

### Preparing audio for the AI

This loads a tool that converts raw audio into a format the AI can understand — similar to how a musician reads sheet music rather than just hearing a song. It standardises things like pitch and volume so the AI can process every recording consistently.

In [11]:
from transformers import WhisperFeatureExtractor

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-tiny")

preprocessor_config.json: 0.00B [00:00, ?B/s]

### Setting up the text converter

This loads a tool that breaks written text into small chunks the AI can work with, and reassembles them back into words when done — like a decoder ring. It's also told upfront that the language is English and the job is transcription, so it's tuned accordingly.

In [12]:
from transformers import WhisperTokenizer

tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-tiny", language="English", task="transcribe", truncation=True)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

### Combining the audio and text tools

This bundles the previous two tools — the audio preparer and the text converter — into a single convenient package, again set to English transcription. Rather than using them separately, we now have one unified tool that handles both ends of the process.

In [ ]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained("openai/whisper-tiny", language="English", task="transcribe")

In [14]:
print(common_voice_dataset["train"][0])

{'path': 'common_voice_en_694779.mp3', 'sentence': 'He paused for a moment to see if the woman knew what the Egyptian pyramids were.', 'age': 'thirties', 'gender': 'male_masculine', 'accents': 'Australian English', 'locale': 'en', 'duration_ms': 4.900125, 'audio': './mdc_downloads/common-voice-v24-english-en-au-subset-fo-0447e8a6_extracted_contents/commonvoice-v24_en-AU/audio_files/common_voice_en_694779.mp3'}


### Standardising the audio quality

This ensures all audio recordings are set to the same sample rate — essentially the same "resolution" of sound. Whisper expects audio at 16,000 samples per second, so any recordings that differ are automatically adjusted to match, like converting all your photos to the same resolution before printing them.

In [15]:
from datasets import Audio

common_voice_dataset = common_voice_dataset.cast_column("audio", Audio(sampling_rate=16000))

### Preparing each recording for training

This is a recipe applied to every audio clip in the dataset. It takes the raw recording, converts it into a visual representation of sound (like a heatmap of frequencies over time), and simultaneously converts the matching transcript into a format the AI can learn from. Both outputs are packaged together so the AI can practice listening and checking its own answers.

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

In [ ]:
common_voice_dataset = common_voice_dataset.map(prepare_dataset, remove_columns=common_voice_dataset.column_names["train"], num_proc=1)

### Loading the AI model

This downloads the Whisper AI model itself — the core "brain" that does the actual speech recognition. We're using the smallest version ("tiny") which is faster and lighter, trading a little accuracy for speed and efficiency.

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")

### Configuring the model's behaviour

This gives the model its instructions: listen for English and convert it to text. The last line removes any pre-set assumptions the model might have had about its output, giving it a clean slate to work from based purely on what it hears.

In [19]:
model.generation_config.language = "english"
model.generation_config.task = "transcribe"

model.generation_config.forced_decoder_ids = None

### Organising recordings into uniform batches

Since audio clips and transcripts vary in length, this acts as a packing tool that groups them together neatly for training. It pads shorter items to match longer ones — like fitting different-sized items into uniform boxes. It also handles some behind-the-scenes housekeeping to ensure the AI only learns from the real content, ignoring any artificial filler added during the packing process.

In [20]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

### Setting up the batch organiser

This puts the packing tool from the previous step into action, connecting it to our processor and model so it knows exactly how to prepare each batch of recordings for training.

In [21]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

In [ ]:
import evaluate

metric = evaluate.load("wer")

### Word error rate

In the following code, we create a function (`compute_metrics`). It is used after each practice run to compares what the AI transcribed against the correct answers and calculates a score called Word Error Rate (`wer`). A lower score is better, as it means fewer words were transcribed incorrectly. It can be thought of as a spell-check that counts how many words the LLM got wrong out as a percentage of the total.

In [ ]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

### Setting the training rules

This is the training plan — a collection of settings that control how the AI learns. It specifies things like how fast it learns, how long it trains for, how often to check its progress, and where to save the results. Think of it as configuring the conditions for a study session: how long to study, how frequently to take practice tests, and where to keep notes.

In [26]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-tiny-au-en",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    warmup_steps=50,
    max_steps=20,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=1,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=5,
    eval_steps=5,
    logging_steps=5,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=True,
)

### Assembling the trainer

This brings everything together into one place — the model, the training and testing data, the batch organiser, the training rules, and the scoring system. Think of it as assembling all the pieces before pressing start: the student, the study material, the exam papers, and the marking criteria, all ready to go.

In [27]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=common_voice_dataset["train"],
    eval_dataset=common_voice_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

In [28]:
processor.save_pretrained(training_args.output_dir)

['./whisper-tiny-au-en/processor_config.json']

In [ ]:
trainer.train()

In [ ]:
kwargs = {
    "dataset": "Common Voice v24 English - en-AU subset for Everything Open 2026",
    "dataset_args": "config: en, split: test",
    "language": "en",
    "model_name": "whisper-tiny-au-en",
    "finetuned_from": "openai/whisper-tiny",
    "tasks": "automatic-speech-recognition",
}

trainer.push_to_hub(**kwargs)

## References

- [https://colab.research.google.com/github/Mozilla-Data-Collective/tutorial-whisper-fine-tuning-australian-EO2026/blob/main/EO2026_teach_whisper_to_speak_Australian.ipynb](https://colab.research.google.com/github/Mozilla-Data-Collective/tutorial-whisper-fine-tuning-australian-EO2026/blob/main/EO2026_teach_whisper_to_speak_Australian.ipynb)
- [https://huggingface.co/blog/fine-tune-whisper](https://huggingface.co/blog/fine-tune-whisper)